# Assignment 04 – Your First Machine Learning Model

## 1. Introduction and Problem Statement

In this assignment, I use the **Medical Appointment No Shows** dataset to build my first machine learning classification models using Python and Scikit-learn.

The dataset contains patient information, appointment details, and whether a patient attended or missed their scheduled appointment.

### Objective

The objective is to predict whether a patient will attend or miss an appointment based on selected patient and appointment-related features.

### Problem Type

This is a **binary classification problem** because the target column, `No-show`, contains two possible classes:

- `No`: Patient attended the appointment.
- `Yes`: Patient missed the appointment.

I will train and evaluate Logistic Regression and Decision Tree classifiers and compare their performance against a majority-class baseline.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving KaggleV2-May-2016.csv to KaggleV2-May-2016.csv


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("KaggleV2-May-2016.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (110527, 14)

Columns:
['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay', 'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show']


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [16]:
# Calculate median using valid ages only
valid_age_median = df.loc[df['Age'] >= 0, 'Age'].median()

# Replace invalid negative age
df.loc[df['Age'] < 0, 'Age'] = valid_age_median

print("Replacement value:", valid_age_median)
print("Negative ages remaining:", (df['Age'] < 0).sum())
print("Minimum age:", df['Age'].min())

Replacement value: 37.0
Negative ages remaining: 0
Minimum age: 0


## Target and Feature Selection

### Target Variable

The target variable is `No-show`, which indicates whether a patient missed their scheduled appointment.

### Selected Features

I selected the following eight features:

- `Age`
- `Gender`
- `Scholarship`
- `Hipertension`
- `Diabetes`
- `Alcoholism`
- `Handcap`
- `SMS_received`

These features represent patient demographics, health-related information, and appointment-related information that may help identify patterns associated with appointment attendance.

### Excluded Features

- `PatientId` and `AppointmentID`: These are identifiers rather than meaningful predictive features. Using raw IDs could introduce irrelevant patterns.
- `Neighbourhood`: Excluded to reduce the risk of geographical or socioeconomic bias.
- `No-show`: Used only as the target variable, never as an input feature, to prevent target leakage.

### Encoding

The `Gender` column is categorical, so I applied one-hot encoding to convert it into numerical columns (`Gender_F` and `Gender_M`) for model training.

In [19]:
features = [
    'Age',
    'Gender',
    'Scholarship',
    'Hipertension',
    'Diabetes',
    'Alcoholism',
    'Handcap',
    'SMS_received'
]

X = df[features]
y = df['No-show']

# One-Hot Encoding
X = pd.get_dummies(X, columns=['Gender'], dtype=int)

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Negative ages:", (X['Age'] < 0).sum())

Features shape: (110527, 9)
Target shape: (110527,)
Negative ages: 0


Train Test Split

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training:", X_train.shape, y_train.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (88421, 9) (88421,)
Testing: (22106, 9) (22106,)


Model Training - Logistic Regression

In [21]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Logistic Regression retrained!")
print("Training accuracy:", model.score(X_train, y_train))
print("Testing accuracy:", model.score(X_test, y_test))

Logistic Regression retrained!
Training accuracy: 0.7977629748589136
Testing accuracy: 0.7992852619198407


Testing VS Training Performance Evaluation

In [23]:
# Evaluate model on training data
train_score = model.score(X_train, y_train)

# Evaluate model on testing data
test_score = model.score(X_test, y_test)

print(f"Training Accuracy: {train_score:.4f}")
print(f"Testing Accuracy:  {test_score:.4f}")

Training Accuracy: 0.7978
Testing Accuracy:  0.7993


### Logistic Regression – Initial Evaluation

The Logistic Regression model achieved:
- Training accuracy: 79.78%
- Testing accuracy: 79.93%

The difference between training and testing accuracy is very small (approximately 0.15 percentage points).

This suggests that the model does not show an obvious overfitting problem. However, its accuracy is only marginally higher than the majority-class baseline, so accuracy alone does not demonstrate that the model is useful.

Training Unrestricted Decision Tree

In [27]:
from sklearn.tree import DecisionTreeClassifier

greedy_model = DecisionTreeClassifier(random_state=42)
greedy_model.fit(X_train, y_train)

print("Decision Tree retrained!")
print("Training accuracy:", greedy_model.score(X_train, y_train))
print("Testing accuracy:", greedy_model.score(X_test, y_test))

Decision Tree retrained!
Training accuracy: 0.8020153583424752
Testing accuracy: 0.7970234325522483


### Decision Tree – Unrestricted Depth

A Decision Tree Classifier without a maximum depth restriction was trained to investigate overfitting.

Results:
- Training accuracy: 80.20%
- Testing accuracy: 79.70%

The training accuracy is higher than the testing accuracy, indicating a small generalisation gap.

Compared with Logistic Regression, the Decision Tree achieved slightly higher training accuracy but slightly lower testing accuracy.

However, the gap is relatively small, so this experiment does not demonstrate severe overfitting. It shows that an unrestricted tree can be more complex, but its actual overfitting must be evaluated using its performance on unseen data.

Baseline calculation

In [29]:
from sklearn.dummy import DummyClassifier

baseline_model = DummyClassifier(strategy='most_frequent')
baseline_model.fit(X_train, y_train)

print("Baseline Training Accuracy:",
      baseline_model.score(X_train, y_train))

print("Baseline Testing Accuracy:",
      baseline_model.score(X_test, y_test))

Baseline Training Accuracy: 0.7977629748589136
Baseline Testing Accuracy: 0.7992852619198407


### Baseline Model Comparison

A DummyClassifier using the most_frequent strategy was trained to establish a baseline.

The baseline achieved a testing accuracy of 79.93%.

Model comparison:
- Logistic Regression: 79.93%
- Decision Tree: 79.70%
- Majority-class baseline: 79.93%

Logistic Regression achieved the same accuracy as the baseline, while the Decision Tree performed slightly worse.

This indicates that neither model has demonstrated meaningful improvement over the majority-class prediction strategy based on accuracy alone.

Since the dataset is imbalanced, additional evaluation metrics such as precision, recall and F1-score are needed to assess the models' ability to identify missed appointments.

Confusion Matrix - Logistic Regression

In [25]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=['No', 'Yes']))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Confusion Matrix:
[[17669     0]
 [ 4437     0]]

Classification Report:
              precision    recall  f1-score   support

          No       0.80      1.00      0.89     17669
         Yes       0.00      0.00      0.00      4437

    accuracy                           0.80     22106
   macro avg       0.40      0.50      0.44     22106
weighted avg       0.64      0.80      0.71     22106



### Confusion Matrix and Classification Report – Logistic Regression

The confusion matrix revealed that the Logistic Regression model predicted every test record as an attended appointment (No).

Results:
- True Negatives: 17,669
- False Positives: 0
- False Negatives: 4,437
- True Positives: 0

The recall for the missed appointment class (Yes) was 0%.

Although the model achieved approximately 80% accuracy, it failed to identify any missed appointments.

This demonstrates why accuracy alone is insufficient for evaluating classification models on imbalanced datasets. Metrics such as precision, recall and F1-score are also necessary.

Classification Threshold Change from 0.5 to 0.3

In [26]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Get probability of each class
y_prob = model.predict_proba(X_test)

# Find the probability column corresponding to "Yes"
yes_index = list(model.classes_).index('Yes')
yes_prob = y_prob[:, yes_index]

# Predict Yes when probability >= 0.3
y_pred_03 = np.where(yes_prob >= 0.3, 'Yes', 'No')

print("Confusion Matrix (Threshold = 0.3):")
print(confusion_matrix(y_test, y_pred_03, labels=['No', 'Yes']))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_03, zero_division=0))

Confusion Matrix (Threshold = 0.3):
[[16250  1419]
 [ 3812   625]]

Classification Report:
              precision    recall  f1-score   support

          No       0.81      0.92      0.86     17669
         Yes       0.31      0.14      0.19      4437

    accuracy                           0.76     22106
   macro avg       0.56      0.53      0.53     22106
weighted avg       0.71      0.76      0.73     22106



### Classification Threshold Experiment

Lowering the classification threshold from 0.5 to 0.3 enabled the Logistic Regression model to identify 625 missed appointments, whereas it identified none at the default threshold. However, this also resulted in 1,419 false positives and reduced accuracy from 79.93% to 76.33%. This demonstrates the trade-off between precision and recall. Although the lower threshold improved the detection of missed appointments, the model still had limited recall (14%) and requires further improvement.

Decision Tree Confusion Matrix

In [28]:
from sklearn.metrics import confusion_matrix, classification_report

tree_pred = greedy_model.predict(X_test)

print("Decision Tree Confusion Matrix:")
print(confusion_matrix(y_test, tree_pred, labels=['No', 'Yes']))

print("\nDecision Tree Classification Report:")
print(classification_report(y_test, tree_pred, zero_division=0))

Decision Tree Confusion Matrix:
[[17542   127]
 [ 4360    77]]

Decision Tree Classification Report:
              precision    recall  f1-score   support

          No       0.80      0.99      0.89     17669
         Yes       0.38      0.02      0.03      4437

    accuracy                           0.80     22106
   macro avg       0.59      0.51      0.46     22106
weighted avg       0.72      0.80      0.72     22106



### Decision Tree – Confusion Matrix and Classification Report

The unrestricted Decision Tree achieved the following results:

- True Negatives: 17,542
- False Positives: 127
- False Negatives: 4,360
- True Positives: 77

The recall for missed appointments (Yes) was approximately 1.74%.

Although the Decision Tree identified 77 missed appointments, it failed to identify most of the actual missed appointments.

This shows that the model's overall accuracy does not adequately represent its ability to detect the minority class.

### Model Evaluation – Final Conclusion

The Logistic Regression and Decision Tree models were evaluated against a majority-class baseline using the same training and testing datasets.

The Logistic Regression model achieved 79.93% test accuracy, equal to the baseline, because it predicted every appointment as attended. The unrestricted Decision Tree achieved 79.70% test accuracy and performed slightly worse than the baseline.

The classification reports revealed that both models struggled to identify missed appointments, which form the minority class. Lowering the Logistic Regression classification threshold from 0.5 to 0.3 improved recall for missed appointments to 14%, but reduced overall accuracy to 76.33%.

The unrestricted Decision Tree showed only a small difference between training and testing accuracy, so substantial overfitting was not observed.

Overall, this experiment demonstrated why accuracy alone is insufficient for evaluating classification models on imbalanced datasets. Further improvements could involve class weighting, feature engineering, better feature selection and evaluation using precision, recall and F1-score.